##Silver - dim_score

### Transformações para a camada silver

####Importando bibliotecas

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

### dim_score
####Consultando dados da tabela bronze

In [0]:
df_score = spark.table("dev_credit_risk.bronze.dim_score")
display(df_score)

#### Início das transformações
#### Retirando os espaços em branco das colunas de tipo string.

In [0]:
for field in df_score.schema.fields:
  if isinstance(field.dataType, StringType):
    df_score = df_score.withColumn(field.name, trim(col(field.name)))

df_score.display()

#### Normalizando os valores

In [0]:
df_score = (
    df_score
    .withColumn(
        "risco",
         F.when(F.upper(F.col("risco")) == "MEDIO", "Médio")
         .otherwise(F.col("risco"))
    )
)
df_score.display()

####Checando o domínio. Valores únicos após normalização

In [0]:
#checando os valores únicos.
display(df_score.select("risco").distinct())

####Renomeando as colunas

####Exibindo o dataframe pronto para ser inserido na tabela delta silver

In [0]:
df_score.display()

####Inserindo os dados na tabela silver

In [0]:
(df_score.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("dev_credit_risk.silver.dim_score")
)

####Consultando os dados com SQL no schema tabela delta silver

In [0]:
%sql
select * from dev_credit_risk.silver.dim_score